# SmolLM2 Integrated Memory V3 — train, evaluate, and publish

**GPU runtime → Run all.** Use `smoke` first for a quick pipeline check and `balanced` for the research run.

This notebook trains the Integrated Memory V3 candidates for **HuggingFaceTB/SmolLM2-135M**, renders the research result tables/plots required by the repository tests, and can publish the validation-selected checkpoint to Hugging Face.

The Hugging Face client is pinned to `huggingface_hub==0.36.2` with `transformers==4.57.6` to avoid mixed-package `DeviceCodeError` failures.

## 1. Setup

In [ ]:
import os, sys, json, subprocess, tempfile, shutil
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

REPO_REF = "main"
REPO = Path(tempfile.mkdtemp(prefix="smollm2-integrated-v3-")) / "TinyCeNN-LM"
subprocess.run(["git", "clone", "--quiet", "https://github.com/vtavakkoli/TinyCeNN-LM.git", str(REPO)], check=True)
subprocess.run(["git", "checkout", REPO_REF], cwd=REPO, check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                "transformers==4.57.6", "huggingface_hub==0.36.2", "datasets>=3,<5",
                "pytest", "nbformat", "pandas", "matplotlib"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "--no-deps"], check=True)

os.environ["PYTHONPATH"] = os.pathsep.join([str(REPO), str(REPO / "src")])
sys.path[:0] = [str(REPO), str(REPO / "src")]

import torch
print("Source:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("Python:", sys.version.split()[0])

## 2. Experiment budget and output directory

In [ ]:
PROFILE = "balanced" # @param ["smoke", "balanced", "extended"]
SAVE_TO_DRIVE = True # @param {type:"boolean"}
SEED = 2028 # @param {type:"integer"}

PROFILES = {
    "smoke": dict(train_contexts="32,64", test_contexts="32,64,128", block_size=8, features=16,
                  train_documents=4, validation_documents=2, test_documents=2, warm_documents=2,
                  warm_steps=2, joint_steps=4, eval_every=2, timing_documents=1, timing_repeats=1, decode_tokens=8),
    "balanced": dict(train_contexts="256,512,1024", test_contexts="256,512,1024,2048", block_size=32, features=64,
                     train_documents=128, validation_documents=16, test_documents=32, warm_documents=16,
                     warm_steps=100, joint_steps=300, eval_every=50, timing_documents=3, timing_repeats=3, decode_tokens=32),
    "extended": dict(train_contexts="512,1024,2048", test_contexts="512,1024,2048,4096", block_size=32, features=64,
                     train_documents=256, validation_documents=32, test_documents=64, warm_documents=32,
                     warm_steps=200, joint_steps=1000, eval_every=100, timing_documents=5, timing_repeats=5, decode_tokens=64),
}

if not torch.cuda.is_available() and PROFILE != "smoke":
    raise RuntimeError("Select a GPU runtime, or use smoke for a CPU pipeline check.")

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/TinyCeNN/smollm2-integrated-v3")
else:
    BASE = Path("/content/smollm2-integrated-v3-results") if Path("/content").exists() else Path.cwd()/"v3-results"

BASE.mkdir(parents=True, exist_ok=True)
run_id = PROFILE + "-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUT, LOG = BASE / run_id, BASE / (run_id + ".log")
config = dict(PROFILES[PROFILE], seed=SEED)
print(json.dumps(config, indent=2))
print("Results:", OUT)

## 3. Optional prior-run exclusions

V3 already excludes the repository's documented prior holdouts/hashes. Add additional manifests only when you want stricter cross-run exclusion.

In [ ]:
ADDITIONAL_MANIFESTS = []
UPLOAD_MANIFESTS = False # @param {type:"boolean"}

if UPLOAD_MANIFESTS:
    from google.colab import files
    folder = Path(tempfile.mkdtemp(prefix="v3-exclusions-"))
    for index, (name, content) in enumerate(files.upload().items()):
        value = json.loads(content)
        if not value.get("document_hashes"):
            raise ValueError(f"{name}: expected a manifest with document_hashes")
        path = folder / f"manifest-{index}.json"
        path.write_bytes(content)
        ADDITIONAL_MANIFESTS.append(str(path))

print("Additional manifests:", len(ADDITIONAL_MANIFESTS))

## 4. CPU preflight

This runs the repository correctness tests before the expensive GPU experiment. The output is captured and printed, so a future failure shows the actual pytest assertion/import error rather than only `CalledProcessError`.

In [ ]:
env = dict(os.environ, CUDA_VISIBLE_DEVICES="", OMP_NUM_THREADS="1", MKL_NUM_THREADS="1")
test = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_integrated_memory.py"],
                      cwd=REPO, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(test.stdout)
if test.returncode:
    raise RuntimeError(f"Integrated-memory preflight failed with exit code {test.returncode}. See pytest output above.")
print("✅ Integrated-memory preflight passed")

## 5. Train and evaluate V3

In [ ]:
command = [sys.executable, "-u", str(REPO / "scripts/benchmark_smollm2_integrated_memory.py"),
           "--output-dir", str(OUT)]
for key, value in config.items():
    command += ["--" + key.replace("_", "-"), str(value)]
for path in ADDITIONAL_MANIFESTS:
    command += ["--exclude-manifest", str(path)]

print(" ".join(command))
try:
    with LOG.open("w") as log:
        with subprocess.Popen(command, cwd=REPO, env=os.environ.copy(), stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line); log.flush()
            status = process.wait()
    if status:
        raise RuntimeError(f"Run failed with code {status}; inspect {LOG}")
except BaseException as error:
    OUT.mkdir(parents=True, exist_ok=True)
    (OUT / "failure_report.json").write_text(json.dumps({"error": str(error), "log": str(LOG)}, indent=2))
    raise
finally:
    if OUT.exists() and LOG.exists():
        shutil.copy2(LOG, OUT / "console.log")
        archive = shutil.make_archive(str(OUT) + "-results", "zip", root_dir=OUT)
        print("Archive:", archive)

## 6. Research results

**Do not remove the `results` tag from the next cell.** The repository's end-to-end preflight test executes that tagged cell offline and verifies that it creates `decision_table.csv` and the per-context `integrated-T*.png` figures.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

OUT = Path(OUT)
summary = pd.read_csv(OUT / "integrated_summary.csv")
selection = json.loads((OUT / "selection.json").read_text())
selected = selection["selected"]

decision_cols = [c for c in [
    "candidate", "context", "test_nll", "test_perplexity", "ppl_ratio",
    "adapted_ppl_ratio", "before_joint_ppl_ratio", "prefill_speedup",
    "decode_speedup", "total_cache_ratio", "beats_both_quality",
    "quality_preserving_efficiency_win", "selected_on_validation"
] if c in summary.columns]

if "matched_control" in summary.columns:
    decisions = summary[summary["matched_control"].notna()].copy()
else:
    decisions = summary[~summary["candidate"].astype(str).str.startswith("attention_") & (summary["candidate"] != "original")].copy()
decisions[decision_cols].sort_values(["candidate", "context"]).to_csv(OUT / "decision_table.csv", index=False)

print("Locked validation selection:", selected)
try:
    display(decisions[decision_cols].sort_values(["candidate", "context"]).reset_index(drop=True))
except NameError:
    print(decisions[decision_cols].sort_values(["candidate", "context"]).to_string(index=False))

for context in sorted(summary["context"].unique()):
    part = decisions[decisions["context"] == context].copy().reset_index(drop=True)
    if part.empty:
        continue
    labels = part["candidate"].astype(str).tolist()
    y = list(range(len(part)))
    fig, axes = plt.subplots(1, 4, figsize=(19, 4.4))
    for ax, col, title, better in [
        (axes[0], "ppl_ratio", "Perplexity vs original", "Lower is better"),
        (axes[1], "adapted_ppl_ratio", "Perplexity vs adapted attention", "Lower is better"),
        (axes[3], "total_cache_ratio", "Total cache / original", "Lower is better"),
    ]:
        vals = part[col] if col in part.columns else pd.Series([float("nan")] * len(part))
        ax.scatter(vals, y)
        ax.axvline(1.0, linestyle="--")
        ax.set_title(title + "\n" + better)
        ax.set_yticks(y); ax.set_yticklabels(labels)
    if "prefill_speedup" in part.columns:
        axes[2].scatter(part["prefill_speedup"], [v - 0.1 for v in y], label="Prefill")
    if "decode_speedup" in part.columns:
        axes[2].scatter(part["decode_speedup"], [v + 0.1 for v in y], marker="s", label="Decode")
    axes[2].axvline(1.0, linestyle="--")
    axes[2].set_title("Original time / candidate time\nHigher is better")
    axes[2].set_yticks(y); axes[2].set_yticklabels(labels); axes[2].legend(fontsize=8)
    fig.suptitle(f"Integrated Memory V3 — context {int(context)}")
    fig.tight_layout()
    fig.savefig(OUT / f"integrated-T{int(context)}.png", dpi=160, bbox_inches="tight")
    plt.close(fig)

history_path = OUT / "training_history.csv"
if history_path.exists():
    history = pd.read_csv(history_path)
    if len(history) and "validation_nll" in history.columns:
        fig, ax = plt.subplots(figsize=(10, 4))
        for candidate, frame in history.groupby("candidate"):
            frame = frame.sort_values("step")
            ax.plot(frame["step"], frame["validation_nll"], marker="o", label=str(candidate))
        ax.set_xlabel("Joint-training step"); ax.set_ylabel("Validation NLL")
        ax.set_title("Integrated Memory V3 joint training"); ax.legend(fontsize=8)
        fig.tight_layout(); fig.savefig(OUT / "joint-training.png", dpi=160, bbox_inches="tight"); plt.close(fig)

print("Wrote:", OUT / "decision_table.csv")
print("Result plots:", sorted(p.name for p in OUT.glob("integrated-T*.png")))

# 7. Publish the selected checkpoint to Hugging Face

This publishes the **exact validation-selected V3 checkpoint**, its pinned SmolLM2 base revision, exact TinyCeNN inference source, benchmark evidence, a loader, and a generated model card.

Before running, add a write-enabled Colab secret named **`HF_TOKEN`**. The upload runs in a fresh Python subprocess so stale in-memory Hub modules cannot cause the `DeviceCodeError` problem.

In [ ]:
# @title Export + publish selected V3 checkpoint
HF_REPO_ID = "vtava/SmolLM2-135M-CeNN-Partition-V3" # @param {type:"string"}
PRIVATE = False # @param {type:"boolean"}

import json, os, shutil, subprocess, sys
from pathlib import Path
import pandas as pd

# Repair the package on disk, but do not import huggingface_hub in this live interpreter.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "--force-reinstall",
                "huggingface_hub==0.36.2"], check=True)

OUT = Path(OUT); REPO = Path(REPO)
selection = json.loads((OUT / "selection.json").read_text())
manifest = json.loads((OUT / "manifest.json").read_text())
report = json.loads((OUT / "integrated_report.json").read_text())
selected = selection["selected"]
record = next(r for r in report["candidates"] if r["candidate"] == selected)
checkpoint = OUT / record["checkpoint"]
if not checkpoint.exists():
    raise FileNotFoundError(f"Selected checkpoint is missing: {checkpoint}")

args = manifest["args"]
PUBLISH = OUT / "huggingface_export"
if PUBLISH.exists(): shutil.rmtree(PUBLISH)
PUBLISH.mkdir(parents=True)
MODEL_FILE = f"{selected}.pt"
shutil.copy2(checkpoint, PUBLISH / MODEL_FILE)

adapter_config = {
    "format": "smollm2-integrated-memory-v3", "candidate": selected, "variant": record["variant"],
    "layers": record["layers"], "base_model": args["base_model"], "base_revision": manifest["model_revision"],
    "tinycenn_source_commit": manifest["source_commit"], "feature_dimension": int(args["features"]),
    "block_size": int(args["block_size"]), "sink_tokens": int(args["sinks"]),
    "validation_nll": record["validation_nll"], "validation_nll_before_joint": record["before_joint_validation_nll"],
    "trainable_parameters": record["trainable_parameters"], "joint_updates": record["joint_updates"],
    "joint_token_presentations": record["joint_token_presentations"],
    "transformers_version": manifest["transformers"], "training_precision": manifest["native_dtype"],
    "source_repository": "https://github.com/vtavakkoli/TinyCeNN-LM"
}
(PUBLISH / "adapter_config.json").write_text(json.dumps(adapter_config, indent=2), encoding="utf-8")

pkg = PUBLISH / "tinycenn_lm"; pkg.mkdir(); (pkg / "__init__.py").write_text("")
for fn in ["integrated_memory.py", "optimized_memory.py"]:
    shutil.copy2(REPO / "src" / "tinycenn_lm" / fn, pkg / fn)
if (REPO / "LICENSE").exists(): shutil.copy2(REPO / "LICENSE", PUBLISH / "LICENSE-TinyCeNN-LM")

loader_lines = [
    "import json", "from pathlib import Path", "import torch",
    "from huggingface_hub import snapshot_download",
    "from transformers import AutoModelForCausalLM, AutoTokenizer",
    "from tinycenn_lm.integrated_memory import restore_student", "",
    f"def load_model(repo_id='{HF_REPO_ID}', device=None, token=None):",
    "    folder = Path(snapshot_download(repo_id, token=token))",
    "    cfg = json.loads((folder / 'adapter_config.json').read_text())",
    "    if torch.cuda.is_available():",
    "        major, _ = torch.cuda.get_device_capability()",
    "        dtype = torch.bfloat16 if major >= 8 else torch.float16",
    "    else: dtype = torch.float32",
    "    base = AutoModelForCausalLM.from_pretrained(cfg['base_model'], revision=cfg['base_revision'], torch_dtype=dtype, attn_implementation='sdpa', token=token).eval().requires_grad_(False)",
    f"    payload = torch.load(folder / '{MODEL_FILE}', map_location='cpu', weights_only=True)",
    "    model = restore_student(base, payload).eval()",
    "    if device is None: device = 'cuda' if torch.cuda.is_available() else 'cpu'",
    "    model = model.to(device)",
    "    tokenizer = AutoTokenizer.from_pretrained(cfg['base_model'], revision=cfg['base_revision'], token=token)",
    "    return model, tokenizer",
]
(PUBLISH / "load_model.py").write_text("\n".join(loader_lines) + "\n", encoding="utf-8")
(PUBLISH / "requirements.txt").write_text(f"torch\ntransformers=={manifest['transformers']}\nhuggingface_hub==0.36.2\n")

bench = PUBLISH / "benchmark"; bench.mkdir()
for fn in ["manifest.json", "selection.json", "integrated_report.json", "validation_summary.csv",
           "integrated_summary.csv", "decision_table.csv", "test_document_nll.csv", "generation_examples.json",
           "training_history.csv", "partition_conservative_history.csv", "joint-training.png"]:
    src = OUT / fn
    if src.exists(): shutil.copy2(src, bench / fn)
for src in OUT.glob("integrated-T*.png"):
    shutil.copy2(src, bench / src.name)

summary = pd.read_csv(OUT / "integrated_summary.csv")
rows = summary[summary["candidate"] == selected].sort_values("context")
card = [
    "---", "language:", "- en", "license: apache-2.0", "library_name: transformers",
    "pipeline_tag: text-generation", f"base_model: {args['base_model']}", "tags:", "- smollm2", "- cenn",
    "- recurrent-memory", "- hybrid-attention", "- memory-efficient", "- experimental", "---", "",
    "# SmolLM2-135M CeNN Partition V3", "",
    f"Validation-selected **`{selected}`** TinyCeNN Integrated Memory V3 checkpoint over `{args['base_model']}`.", "",
    "## Architecture", "", f"- Variant: `{record['variant']}`", f"- Replaced attention layers: `{record['layers']}`",
    f"- Feature dimension: `{args['features']}`", f"- Block size: `{args['block_size']}`", f"- Sink tokens: `{args['sinks']}`",
    f"- Exact base revision: `{manifest['model_revision']}`", f"- TinyCeNN source commit: `{manifest['source_commit']}`", "",
    "The remaining Transformer layers retain standard attention. This is a partial hybrid research checkpoint, not a fully attention-free model.", "",
    "## Held-out evaluation", "",
    "| Context | Perplexity | PPL ratio vs original | Cache ratio | Prefill speedup | Decode speedup |",
    "|---:|---:|---:|---:|---:|---:|",
]
for _, r in rows.iterrows():
    card.append(f"| {int(r['context'])} | {r['test_perplexity']:.3f} | {r['ppl_ratio']:.4f} | {r['total_cache_ratio']:.4f} | {r['prefill_speedup']:.3f}× | {r['decode_speedup']:.3f}× |")
card += [
    "", "Ratios below 1.0 are better for perplexity/cache. Speedups above 1.0 are faster. Current PyTorch CeNN kernels are experimental and are not yet optimized like GPU SDPA.", "",
    "## Load", "", "```python", "import sys", "from huggingface_hub import snapshot_download",
    f"folder = snapshot_download('{HF_REPO_ID}')", "sys.path.insert(0, folder)", "from load_model import load_model",
    f"model, tokenizer = load_model('{HF_REPO_ID}')", "```", "",
    "## Reproducibility", "", f"- Validation NLL: `{record['validation_nll']}`",
    f"- Before-joint validation NLL: `{record['before_joint_validation_nll']}`", f"- Joint updates: `{record['joint_updates']}`",
    f"- Trainable TinyCeNN parameters: `{record['trainable_parameters']}`", f"- Training precision: `{manifest['native_dtype']}`",
    "- Full benchmark evidence is under `benchmark/`.", "",
    "## Limitations", "", "This checkpoint uses a limited held-out set and one training seed. It does not establish universal superiority over Transformer attention. Qualitative generations are not benchmark evidence.", "",
    "## Source and licenses", "", "TinyCeNN-LM: https://github.com/vtavakkoli/TinyCeNN-LM",
    "Base model: https://huggingface.co/HuggingFaceTB/SmolLM2-135M",
    "SmolLM2 is Apache-2.0. TinyCeNN-LM source is MIT; its license copy is included as `LICENSE-TinyCeNN-LM`.",
]
(PUBLISH / "README.md").write_text("\n".join(card) + "\n", encoding="utf-8")

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None
if not HF_TOKEN:
    raise RuntimeError("Add a write-enabled Colab Secret named HF_TOKEN, enable notebook access, then rerun this cell.")

upload_script = """
import os, huggingface_hub
from huggingface_hub import HfApi
print('huggingface_hub:', huggingface_hub.__version__)
token=os.environ['HF_TOKEN']; repo=os.environ['HF_REPO_ID']; folder=os.environ['HF_EXPORT_DIR']
api=HfApi(token=token)
api.create_repo(repo_id=repo, repo_type='model', private=os.environ.get('HF_PRIVATE','false')=='true', exist_ok=True, token=token)
api.upload_folder(repo_id=repo, repo_type='model', folder_path=folder, token=token, commit_message='Publish TinyCeNN Integrated Memory V3 checkpoint')
print('https://huggingface.co/' + repo)
"""
env = os.environ.copy(); env.update(HF_TOKEN=HF_TOKEN, HF_REPO_ID=HF_REPO_ID, HF_EXPORT_DIR=str(PUBLISH), HF_PRIVATE=str(PRIVATE).lower())
uploaded = subprocess.run([sys.executable, "-c", upload_script], env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(uploaded.stdout)
if uploaded.returncode:
    raise RuntimeError(f"Hugging Face upload failed with exit code {uploaded.returncode}. See output above.")
print(f"✅ Published exact selected checkpoint: {selected} -> https://huggingface.co/{HF_REPO_ID}")